In [ ]:
# Change settings here, then Run All (Kaggle: Internet + 2 GPUs).
from pathlib import Path
import os

# Puzzle selection. START_STATE_OVERRIDE requires exactly one selected puzzle.
PUZZLE_ID_START = 1020
PUZZLE_ID_END = 1020
INDICES_URL = None  # Optional newline-separated puzzle IDs
RANDOM_PUZZLE = False
START_STATE_OVERRIDE = None  # Optional state accepted by CayleyPy

# Search parameters. Use a smaller beam / max steps for a bounded experiment.
BEAM_WIDTH = 1048576  # Integer or "max"
MAX_STEPS = 140
BEAM_MODE = "simple"
USE_CHILD_SCORES = True
RETURN_PATH = True
NUM_GPUS = 2  # None uses every visible GPU

# Cube555 model: weight 1 uses Transformer, 0 uses MLP, between 0 and 1 blends both.
MODEL_ROOT = None  # None downloads the pinned GitHub model bundle
COMPETITION_ROOT = None  # None downloads the pinned puzzle data
CHECKPOINT_PATH = "q555_f1_bell2k.pt"
RESMLP_CHECKPOINT_PATH = "q555_2k_BEST.pt"
LAYOUT_PATH = "piece_layout_555.json"
TRANSFORMER_WEIGHT = 0.8

# Collection, BFS touch, and reflection.
SOLUTION_MODE = "collect"  # "first" or "collect"
COLLECT_EXTRA_DEPTHS = 1
MAX_COLLECTED_SOLUTIONS = 2000
TOUCH_BFS_RADIUS = 5
TOUCH_BFS_MAX_ENTRIES = 30_000_000
REFLECT_MODE = "off"  # "off", "after_original", or "only"
REFLECT_SOURCE_CSV = None  # Required when REFLECT_MODE is "only"

# Runtime, caches, and output.
RUN_TIMEOUT_SECONDS = 12 * 3600
AUTO_TUNE_PROFILES = True  # exact local cache -> matching GitHub profile -> calibrate
CALIBRATION_SECONDS = 90
CALIBRATION_PIPELINE_SECONDS = 1_800
WORK_ROOT = Path(os.environ.get("MULTIGPUBEAMSEARCH_HOME", Path.home() / ".cache/multigpubeamsearch/cube555"))
os.environ["MULTIGPUBEAMSEARCH_HOME"] = str(WORK_ROOT)
CACHE_DIR = WORK_ROOT / "native_cache"
OUTPUT_ROOT = Path.cwd() / "cube555_runs"

# Kaggle → Cloudflare → GitHub result publishing.
PUBLISH_RESULTS = True
AUTHOR = "trydotatwo"  # change this when making your own notebook copy
KAGGLE_OWNER = "trydotatwo"
KAGGLE_SLUG = "cube555-native-2xt4-blend"
KAGGLE_VERSION = 174  # must match the saved Kaggle version for this run


# Cube555 through CayleyPy
Edit the first cell, then Run All. The notebook installs the pinned MultiGPUBeamSearch/CayleyPy integration and downloads model weights and puzzle data from GitHub. Profiles are selected automatically for the requested frontier and available GPUs. Every returned solution is replayed before saving. On a two-GPU Kaggle run, valid results are sent through Cloudflare to cayleypy-beam-results; the payload, status, and receipts are saved under `cube555_runs`.

Profile selection is automatic: exact local cache, then an exact compatible GitHub profile, then calibration for the requested frontier. A profile for another beam is never reused.

In [ ]:
# First-time installation (also runs automatically with Run All).
from urllib.request import urlretrieve
from runpy import run_path
from tempfile import TemporaryDirectory

with TemporaryDirectory() as setup_dir:
    setup_file, _ = urlretrieve("https://raw.githubusercontent.com/TryDotAtwo/MultiGPUBeamSearch/0e8d1b1bee2569d75960e5bf00b655552c632f9a/kaggle/cube555_github/setup.py", setup_dir + "/setup.py")
    run_path(setup_file)


In [ ]:
import cayleypy as cp
import multigpubeamsearch as beam
from multigpubeamsearch import cube555


In [ ]:
if not 0.0 <= TRANSFORMER_WEIGHT <= 1.0:
    raise ValueError("TRANSFORMER_WEIGHT must be between 0 and 1")
if START_STATE_OVERRIDE is not None and PUZZLE_ID_START != PUZZLE_ID_END and not RANDOM_PUZZLE:
    raise ValueError("START_STATE_OVERRIDE can only be used with one selected puzzle")

puzzle_ids = cube555.select_puzzles(PUZZLE_ID_START, PUZZLE_ID_END,
    indices_url=INDICES_URL, random_choice=RANDOM_PUZZLE)
if START_STATE_OVERRIDE is not None and len(puzzle_ids) != 1:
    raise ValueError("START_STATE_OVERRIDE requires exactly one selected puzzle")
puzzle = cube555.load(puzzle_ids[0], cache_dir=WORK_ROOT,
    model_root=MODEL_ROOT, competition_root=COMPETITION_ROOT)
graph = cp.CayleyGraph(puzzle.definition, device="cpu", random_seed=42)
predictor = puzzle.blend(graph, transformer_weight=TRANSFORMER_WEIGHT,
    transformer_checkpoint=CHECKPOINT_PATH, mlp_checkpoint=RESMLP_CHECKPOINT_PATH,
    layout_path=LAYOUT_PATH)
beam.enable_native(beam.NativeOptions(num_gpus=NUM_GPUS, cache_dir=CACHE_DIR,
    timeout_seconds=RUN_TIMEOUT_SECONDS, autotune=AUTO_TUNE_PROFILES,
    calibration_seconds=CALIBRATION_SECONDS,
    calibration_pipeline_seconds=CALIBRATION_PIPELINE_SECONDS))


In [ ]:
import json

with beam.NativeSession() as session:
    for puzzle_id in puzzle_ids:
        puzzle = cube555.load(puzzle_id, cache_dir=WORK_ROOT,
            model_root=MODEL_ROOT, competition_root=COMPETITION_ROOT)
        start_state = puzzle.start_state if START_STATE_OVERRIDE is None else START_STATE_OVERRIDE
        result = graph.beam_search(
            start_state=start_state, predictor=predictor, backend="native",
            beam_width=BEAM_WIDTH, max_steps=MAX_STEPS, beam_mode=BEAM_MODE,
            use_child_scores=USE_CHILD_SCORES, return_path=RETURN_PATH,
            solution_mode=SOLUTION_MODE, collect_extra_depths=COLLECT_EXTRA_DEPTHS,
            max_collected_solutions=MAX_COLLECTED_SOLUTIONS,
            bfs_radius=TOUCH_BFS_RADIUS, bfs_max_entries=TOUCH_BFS_MAX_ENTRIES,
            reflect_mode=REFLECT_MODE,
            reflection_source=cube555.source_solution(REFLECT_SOURCE_CSV, puzzle_id, graph),
        )
        report = cube555.finish_result(result, graph, start_state,
            publish=PUBLISH_RESULTS, directory=OUTPUT_ROOT, puzzle_id=puzzle_id,
            predictor=predictor, author=AUTHOR, kaggle_owner=KAGGLE_OWNER,
            kaggle_slug=KAGGLE_SLUG, kaggle_version=KAGGLE_VERSION,
            notebook_path=Path("/kaggle/working/__notebook__.ipynb"),
            solver_commit=os.environ["MULTIGPUBEAMSEARCH_SOURCE_COMMIT"])
        saved = report["saved"]
        print(puzzle_id, "solutions:", len(result.solutions),
              "best length:", result.path_length if result.path_found else None,
              "saved:", saved, "publication:", report["publication_status"])
        profile = result.native_metadata.get("profile", {})
        inference = profile.get("inference_calibration", {})
        pipeline = profile.get("pipeline_calibration", {})
        source = ("local cache" if inference.get("cache_hit") else
                  "GitHub profile" if inference.get("shared_profile_validated") else
                  "calibrated now" if profile.get("inference_autotuned") else "library default")
        print(f"Profile: {source}; batch={inference.get('parent_batch')}; "
              f"frontier={BEAM_WIDTH}; pipeline={pipeline.get('selection', 'cached/default')}")
        elif PUBLISH_RESULTS:
            print("No replay-valid solution found in this run; nothing was submitted.")
